# 第6章 回帰

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍の入力番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/6/
- 演習の解答: https://ml.kano.ac/solutions/6/

## 単回帰

### 勉強時間と試験の点数の予測

**入力 6.1**　単回帰モデルの学習と予測

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# データの作成（勉強時間と試験の点数）
np.random.seed(42)
X = np.arange(1, 21).reshape(-1, 1).astype(float)
# 傾き3.5、切片30の直線にノイズを加えてyを作成
y = 3.5 * X.ravel() + 30 + np.random.randn(20) * 5

# データの分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# モデルの学習
model = LinearRegression()
model.fit(X_train, y_train)

# パラメータの確認
print(f"傾き (a): {model.coef_[0]:.2f}")
print(f"切片 (b): {model.intercept_:.2f}")
print(f"回帰式: y = {model.coef_[0]:.2f}x + {model.intercept_:.2f}")

# テストデータでの予測
y_pred = model.predict(X_test)
for xi, yt, yp in zip(X_test.ravel(), y_test, y_pred):
    print(f"  勉強時間 {xi:.0f}h → 実際: {yt:.1f}, 予測: {yp:.1f}")

### 回帰直線の可視化

**入力 6.2**　散布図と回帰直線の可視化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 可視化
plt.figure(figsize=(5.8, 3.6))
plt.scatter(X_train, y_train, color="blue", label="訓練データ")
plt.scatter(X_test, y_test, color="green", marker="s",
            label="テストデータ")

# 回帰直線
X_line = np.linspace(0, 22, 100).reshape(-1, 1)
y_line = model.predict(X_line)
plt.plot(X_line, y_line, color="red", linewidth=2,
         label="回帰直線")

plt.xlabel("勉強時間")
plt.ylabel("試験の点数")
plt.title("単回帰")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 重回帰

### 住宅価格データでの実装

**入力 6.3**　住宅価格データの作成

In [ ]:
import pandas as pd
import numpy as np

# 住宅データの作成
np.random.seed(42)
n = 50
data = pd.DataFrame({
    "面積": np.random.randint(40, 130, n),
    "築年数": np.random.randint(0, 35, n),
    "駅徒歩": np.random.randint(1, 25, n)
})
data["価格"] = (
    35 * data["面積"]
    - 30 * data["築年数"]
    - 15 * data["駅徒歩"]
    + 500
    + np.random.randn(n) * 100
)

print(data.head())
print(f"\nデータ数: {len(data)}")

### 重回帰モデルの学習と係数の解釈

**入力 6.4**　重回帰モデルの学習と係数の確認

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 特徴量と目的変数の分離
X = data[["面積", "築年数", "駅徒歩"]]
y = data["価格"]

# データの分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# モデルの学習
model = LinearRegression()
model.fit(X_train, y_train)

# 係数の確認
coef_df = pd.DataFrame({
    "特徴量": X.columns,
    "係数": model.coef_
})
print(coef_df)
print(f"\n切片: {model.intercept_:.2f}")
print(f"訓練データ R^2: {model.score(X_train, y_train):.4f}")
print(f"テストデータ R^2: {model.score(X_test, y_test):.4f}")

### 予測値と実測値の比較

**入力 6.5**　実測値と予測値の散布図

In [ ]:
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)

plt.figure(figsize=(2.4, 2.4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         "r--", linewidth=2, label="y = x")
plt.xlabel("実際の価格")
plt.ylabel("予測価格")
plt.title("実際の値 vs 予測値")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 多項式回帰

### PolynomialFeaturesによる特徴量の変換

**入力 6.6**　`PolynomialFeatures`による特徴量変換

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

X = np.array([[2], [3], [4]])

# 2次の多項式特徴量を生成
poly = PolynomialFeatures(degree=2,
    include_bias=False)
X_poly = poly.fit_transform(X)

print("元の特徴量:")
print(X)
print("\n多項式特徴量 (x, x^2):")
print(X_poly)
print(f"\n特徴量名: {poly.get_feature_names_out()}")

### 非線形データへの適用

**入力 6.7**　次数の異なる多項式回帰の比較

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error

# 非線形データの生成
np.random.seed(42)
X = np.sort(np.random.rand(30, 1) * 2 * np.pi, axis=0)
y = np.sin(X).ravel() + np.random.randn(30) * 0.2

# 次数ごとにモデルを学習
degrees = [1, 3, 5, 15]
fig, axes = plt.subplots(2, 2, figsize=(5.8, 4.6))

X_plot = np.linspace(0, 2 * np.pi, 200).reshape(-1, 1)

for ax, degree in zip(axes.ravel(), degrees):
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X)
    X_plot_poly = poly.transform(X_plot)

    model = LinearRegression()
    model.fit(X_poly, y)

    y_plot = model.predict(X_plot_poly)
    rmse = np.sqrt(mean_squared_error(y, model.predict(X_poly)))

    ax.scatter(X, y, color="blue", alpha=0.6, s=20)
    ax.plot(X_plot, y_plot, color="red", linewidth=2)
    ax.set_title(f"次数 {degree} (RMSE={rmse:.4f})")
    ax.set_ylim(-2, 2)

plt.tight_layout()
plt.show()

## 正則化（リッジ回帰、ラッソ回帰）

### リッジ回帰（L2正則化）

**入力 6.8**　線形回帰とリッジ回帰の比較

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error

# 非線形データの生成
np.random.seed(42)
X = np.sort(np.random.rand(30, 1) * 10, axis=0)
y = np.sin(X).ravel() + np.random.randn(30) * 0.3

# 高次の多項式特徴量を生成（過学習しやすい状況を作る）
poly = PolynomialFeatures(degree=10, include_bias=False)
X_poly = poly.fit_transform(X)

# データの分割
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.3, random_state=42
)

# 訓練データを基準に特徴量を標準化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 通常の線形回帰
lr = LinearRegression()
lr.fit(X_train, y_train)
print("【通常の線形回帰】")
print(f"訓練 RMSE: "
      f"{np.sqrt(mean_squared_error(y_train, lr.predict(X_train))):.4f}")
print(f"テスト RMSE: "
      f"{np.sqrt(mean_squared_error(y_test, lr.predict(X_test))):.4f}")

# リッジ回帰
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
print("\n【リッジ回帰 (alpha=1.0)】")
print(f"訓練 RMSE: "
      f"{np.sqrt(mean_squared_error(y_train, ridge.predict(X_train))):.4f}")
print(f"テスト RMSE: "
      f"{np.sqrt(mean_squared_error(y_test, ridge.predict(X_test))):.4f}")

### ラッソ回帰（L1正則化）

**入力 6.9**　ラッソ回帰の学習と係数の確認

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

# ラッソ回帰
lasso = Lasso(alpha=0.1, max_iter=100000)
lasso.fit(X_train, y_train)
print("【ラッソ回帰 (alpha=0.1)】")
print(f"訓練 RMSE: "
      f"{np.sqrt(mean_squared_error(y_train, lasso.predict(X_train))):.4f}")
print(f"テスト RMSE: "
      f"{np.sqrt(mean_squared_error(y_test, lasso.predict(X_test))):.4f}")

# 係数の確認（0になった係数の数）
n_zero = np.sum(lasso.coef_ == 0)
print(f"\n全係数の数: {len(lasso.coef_)}")
print(f"0になった係数の数: {n_zero}")
print(f"0でない係数の数: {len(lasso.coef_) - n_zero}")

## 回帰モデルの評価指標

### scikit-learnによる実装

**入力 6.10**　回帰の評価指標の計算

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, r2_score

# データの準備
np.random.seed(42)
n = 80
data = pd.DataFrame({
    "面積": np.random.randint(40, 130, n),
    "築年数": np.random.randint(0, 35, n),
    "駅徒歩": np.random.randint(1, 25, n)
})
data["価格"] = (
    35 * data["面積"] - 30 * data["築年数"]
    - 15 * data["駅徒歩"] + 500
    + np.random.randn(n) * 150
)

X = data[["面積", "築年数", "駅徒歩"]]
y = data["価格"]

# データの分割と学習
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 評価指標の計算
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("【回帰モデルの評価結果】")
print(f"MSE  : {mse:>12.2f} (万円^2)")
print(f"RMSE : {rmse:>12.2f} (万円)")
print(f"MAE  : {mae:>12.2f} (万円)")
print(f"R^2  : {r2:>12.4f}")

### 複数モデルの比較

**入力 6.11**　複数モデルの評価指標による比較

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, r2_score

# 複数のモデルを比較
models = {
    "LinearRegression": LinearRegression(),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0),
    "Ridge (alpha=10.0)": Ridge(alpha=10.0),
    "Lasso (alpha=1.0)": Lasso(alpha=1.0),
    "Lasso (alpha=10.0)": Lasso(alpha=10.0),
}

results = []
for name, m in models.items():
    m.fit(X_train, y_train)
    y_p = m.predict(X_test)
    results.append({
        "モデル": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_p)),
        "MAE": mean_absolute_error(y_test, y_p),
        "R^2": r2_score(y_test, y_p)
    })

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

## 演習問題

### 演習 6-1: tips データセットで単回帰分析

tips データセットを使って、食事代（`total_bill`）からチップ（`tip`）を予測する単回帰分析を実装してください。

**タスク**

1. tips データセットを読み込み、先頭 5 行を表示する
2. 特徴量 `X = tips[['total_bill']]`、ラベル `y = tips['tip']` を準備する
3. データを「訓練 80%、テスト 20%」（`random_state=42`）に分割する
4. 線形回帰モデルを学習する
5. 学習された **傾き（係数）** と **切片** を表示する
6. テストデータで予測し、**RMSE** と **R²** を計算して表示する
7. 訓練データ（青）、テストデータ（緑）、回帰直線（赤）を 1 つの散布図で可視化する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. データの読み込みと先頭 5 行の表示

# 2. 特徴量 X とラベル y の設定

# 3. 訓練・テストに分割（8:2, random_state=42）

# 4. モデルの作成と学習

# 5. 傾きと切片の表示

# 6. テストデータで予測し、RMSE と R² を表示

# 7. 訓練データ・テストデータ・回帰直線を散布図で可視化

[解答例を見る](https://ml.kano.ac/solutions/6/#solution-6-1)

### 演習 6-2: tips データセットで重回帰分析

tips データセットを使って、食事代（`total_bill`）とグループサイズ（`size`）の 2 つの特徴量から、チップ（`tip`）を予測する重回帰分析を実装してください。

**タスク**

1. 特徴量 `X = tips[['total_bill', 'size']]`、ラベル `y = tips['tip']` を準備する
2. データを「訓練 80%、テスト 20%」（`random_state=42`）に分割する
3. 重回帰モデルを学習する
4. 各特徴量の **係数** と **切片** を表示する
5. テストデータで予測し、**RMSE** と **R²** を計算して表示する
6. 横軸に実測値、縦軸に予測値をとった散布図を描き、`y = x` の理想線（赤）も重ねて表示する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

tips = sns.load_dataset('tips')

# 1. 特徴量 X とラベル y の設定

# 2. 訓練・テストに分割（8:2, random_state=42）

# 3. モデルの作成と学習

# 4. 各特徴量の係数と切片の表示

# 5. テストデータで予測し、RMSE と R² を表示

# 6. 実測値 vs 予測値の散布図と理想線（y = x）

[解答例を見る](https://ml.kano.ac/solutions/6/#solution-6-2)

### 演習 6-3: 4 つの評価指標と過学習チェック

演習 6-2 と同じ重回帰モデル（`total_bill`, `size` → `tip`）について、訓練データとテストデータの両方で 4 つの評価指標（MSE, RMSE, MAE, R²）を計算し、過学習が起きていないか判定してください。

**タスク**

1. データを「訓練 80%、テスト 20%」（`random_state=42`）に分割する
2. 重回帰モデルを学習する
3. 訓練データに対する予測値 `y_train_pred` と、テストデータに対する予測値 `y_test_pred` を計算する
4. 訓練データの MSE, RMSE, MAE, R² を表示する
5. テストデータの MSE, RMSE, MAE, R² を表示する
6. 訓練 R² とテスト R² の差から、以下の基準で過学習を判定する
    - 差 < 0.05 → 「過学習なし（良好）」
    - 0.05 ≤ 差 < 0.15 → 「若干の過学習あり」
    - 0.15 ≤ 差 → 「過学習が発生」

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

tips = sns.load_dataset('tips')
X = tips[['total_bill', 'size']]
y = tips['tip']

# 1. 訓練・テストに分割（8:2, random_state=42）

# 2. モデルの作成と学習

# 3. 訓練データとテストデータそれぞれの予測値を計算

# 4. 訓練データの MSE, RMSE, MAE, R² を表示

# 5. テストデータの MSE, RMSE, MAE, R² を表示

# 6. 訓練 R² とテスト R² の差から過学習を判定

[解答例を見る](https://ml.kano.ac/solutions/6/#solution-6-3)

### 演習 6-4: 多項式回帰の次数比較と過学習の観察

人工的に生成した非線形データに次数の異なる多項式回帰を当てはめて最適な次数を選び、次数を上げすぎると過学習が起きることを実験的に確認してください。

**タスク**

1. 以下のコードで非線形データを生成する（コピペで OK）

In [ ]:
np.random.seed(0)
X = np.sort(np.random.rand(80, 1) * 10, axis=0)
y = 0.3 * X.squeeze() ** 2 - 2 * X.squeeze() + 5 + np.random.randn(80) * 3

2. データを「訓練 80%、テスト 20%」（`random_state=42`）に分割する
3. `degrees = [1, 2, 3, 5, 10]` の各次数について、多項式回帰モデルを学習する
4. 各次数について、「訓練 R²」と「テスト R²」を計算し、表形式で表示する
5. テスト R² が最大の次数（最も汎化性能が高い）を確認し、最適な次数として選ぶ
6. 各次数の予測曲線を 2×3 のサブプロットで可視化する（訓練データ：青、テストデータ：緑、予測曲線：赤）
7. 次数を上げすぎると訓練 R² とテスト R² がどう変化するか考察する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. データ生成
np.random.seed(0)
X = np.sort(np.random.rand(80, 1) * 10, axis=0)
y = 0.3 * X.squeeze() ** 2 - 2 * X.squeeze() + 5 + np.random.randn(80) * 3

degrees = [1, 2, 3, 5, 10]

# 2. 訓練・テストに分割（8:2, random_state=42）

# 3. 各次数で多項式回帰モデルを学習

# 4. 各次数の訓練 R² とテスト R² を表形式で表示

# 5. テスト R² が最大の次数を表示

# 6. 各次数の予測曲線を 2×3 のサブプロットで可視化

[解答例を見る](https://ml.kano.ac/solutions/6/#solution-6-4)

### 演習 6-5: 正則化の効果を確認する

演習 6-4 と同じ人工データに対して、次数 10 の多項式回帰にリッジ回帰とラッソ回帰を適用し、正則化の効果を確認してください。

**タスク**

1. データを「訓練 80%、テスト 20%」（`random_state=42`）に分割する
2. 次数 10 の `PolynomialFeatures` で特徴量を変換する
3. 通常の `LinearRegression`、`Ridge(alpha=0.1)`、`Lasso(alpha=0.1)` の 3 つのモデルを学習する
4. 各モデルのテストデータでの **RMSE** と **R²** を比較する
5. 各モデルの係数の大きさ（絶対値の最大値）を比較し、正則化の効果を確認する

**ヒント**：次数 10 では x^10 のような高次の項の値が非常に大きくなるため、`StandardScaler` で標準化してから学習すると結果が安定します。

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# データの生成（演習 6-4 と同じ）
np.random.seed(0)
X = np.sort(np.random.rand(80, 1) * 10, axis=0)
y = 0.3 * X.squeeze() ** 2 - 2 * X.squeeze() + 5 + np.random.randn(80) * 3

# 1. 訓練・テストに分割（8:2, random_state=42）

# 2. 次数 10 の PolynomialFeatures で特徴量を変換（標準化も行う）

# 3. LinearRegression, Ridge(alpha=0.1), Lasso(alpha=0.1) を学習

# 4. 各モデルのテストデータでの RMSE と R² を比較

# 5. 各モデルの係数の絶対値の最大値を比較

[解答例を見る](https://ml.kano.ac/solutions/6/#solution-6-5)